In [11]:
import polars as pl
import openpyxl
import glob
import os
import re
import datetime as _dt
from datetime import datetime, date, time
from collections import defaultdict
from openpyxl.utils import column_index_from_string, get_column_letter
from IPython.display import display

In [12]:
first_glob = os.path.expanduser("~").replace("\\", "/")

exclibur_path = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Performance All Sites.xlsx'
output_req    = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Req.csv'
output_ou_req = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/OU_Req_File.csv'
parquet_path  = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/excalibur_raw.parquet'
input_ou_mail = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_OU_MAIL'

In [13]:
date_from  = _dt.date(2026, 8, 1)
date_to    = _dt.date(2026, 8, 30)
output_cmp = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Req_Comparison.xlsx'

df = pl.read_excel(exclibur_path, sheet_name="Requierd Hours", engine="calamine", has_header=False)

raw_headers = [str(val).split(" ")[0] for val in df.row(0)]
seen = defaultdict(int)
new_headers = []
for h in raw_headers:
    count = seen[h]
    new_headers.append(h if count == 0 else f"{h}_{count}")
    seen[h] += 1

mtd_index = next((i for i, h in enumerate(new_headers) if h == "MTD"), None)
if mtd_index is None:
    raise ValueError(f"MTD column not found. Headers: {new_headers}")

cols_to_keep   = df.columns[:mtd_index + 1]
rename_mapping = dict(zip(cols_to_keep, new_headers[:mtd_index + 1]))
result_df      = df.slice(1, 980).select(cols_to_keep).rename(rename_mapping)

def _parse_interval(val) -> str | None:
    if val is None:
        return None
    s = str(val).strip()

    for fmt in ('%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M'):
        try: return _dt.datetime.strptime(s, fmt).strftime('%H:%M:%S')
        except ValueError: pass

    for fmt in ('%H:%M:%S', '%H:%M'):
        try: return _dt.datetime.strptime(s, fmt).strftime('%H:%M:%S')
        except ValueError: pass

    for fmt in ('%I:%M:%S %p', '%I:%M %p'):
        try: return _dt.datetime.strptime(s, fmt).strftime('%H:%M:%S')
        except ValueError: pass

    try:
        f = float(s)
        if 0.0 <= f < 1.0:
            total_sec = round(f * 86400)
            return f'{total_sec // 3600:02d}:{(total_sec % 3600) // 60:02d}:{total_sec % 60:02d}'
    except ValueError:
        pass

    return None

result_df = result_df.with_columns([
    pl.col("PSP").replace_strict({"Chat Non-Lodging": "Non-Lodging chat"}, default=pl.col("PSP")),
    pl.col("Interval")
      .map_elements(_parse_interval, return_dtype=pl.String)
      .alias("Interval"),
])

data_cols = [c for c in result_df.columns if c not in ("PSP", "Site", "Interval")]

def to_numeric_expr(col_name: str, dtype) -> pl.Expr:
    dtype_str = str(dtype)
    if dtype_str.startswith("Datetime") or dtype_str.startswith("Time"):
        return (
            pl.col(col_name).dt.hour()
            + pl.col(col_name).dt.minute() / 60
            + pl.col(col_name).dt.second() / 3600
            + pl.col(col_name).dt.microsecond() / 3_600_000_000
        ).alias(col_name)
    else:
        return pl.col(col_name).cast(pl.Float64, strict=False).alias(col_name)

result_df = result_df.with_columns([
    to_numeric_expr(c, result_df.schema[c]) for c in data_cols
]).filter(
    pl.col("PSP").is_in(["Lodging chat", "Non-Lodging chat"])
).sort("PSP", maintain_order=True)

print(f"[Req.csv] Shape: {result_df.shape} | PSP: {result_df['PSP'].unique(maintain_order=True).to_list()}")

ROW_START = 5
ROW_END   = 53
DAYS            = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
METRICS         = ['Req W', 'Prov', 'DIFF']
DAY_METRIC_COLS = [f'{d}_{m}' for d in DAYS for m in METRICS]
DAY_OFFSET      = {'Mon': 0, 'Tue': 1, 'Wed': 2, 'Thu': 3, 'Fri': 4, 'Sat': 5, 'Sun': 6}

LG_SHEET_CONFIG = {
    'VNM - OU'  : ('E', 'Y'), 'Kol - OU'  : ('E', 'Y'),
    'Pune - OU' : ('E', 'Y'), 'Gobal - OU': ('G', 'AA'), 'Egypt - OU': ('E', 'Y'),
}
NL_SHEET_CONFIG = {
    'VNM - OU'  : ('E', 'Y'), 'Kol - OU'  : ('E', 'Y'),
    'Pune - OU' : ('E', 'Y'), 'Gobal - OU': ('F', 'Z'),  'Egypt - OU': ('E', 'Y'),
}
OU_TO_SITE = {
    'VNM - OU'   : 'Concentrix (Ho Chi Minh City)', 'Kol - OU'   : 'Concentrix (Kolkata)',
    'Pune - OU'  : 'Concentrix (Pune)',              'Gobal - OU' : 'Concentrix (Global)',
    'Global - OU': 'Concentrix (Global)',             'Egypt - OU' : 'Concentrix (Cairo)',
}
ou_cache_path = f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/ou_mail_raw_cache.parquet'

def _extract_week(path):
    m = re.search(r'(\d{2})_(\d{2})_(\d{2})', os.path.basename(path))
    if not m: return None
    return _dt.date(int(m[3]) + 2000, int(m[1]), int(m[2]))

def _find_all(folder, keyword):
    matches = glob.glob(os.path.join(folder, f'*{keyword}*.xlsx'))
    if not matches: raise FileNotFoundError(f'No file matched "{keyword}" in: {folder}')
    available = []
    for p in matches:
        try: open(p, 'rb').close(); available.append(p)
        except (IOError, OSError): print(f'  Skipped (cloud-only): {os.path.basename(p)}')
    available.sort(key=lambda p: _extract_week(p) or _dt.date.fromtimestamp(os.path.getmtime(p)))
    return available

def _resolve_sheet(sheetnames, target):
    if target in sheetnames: return target
    def _norm(s): return s.lower().replace(' ', '').replace('-', '')
    t = _norm(target)
    for s in sheetnames:
        if _norm(s) == t: return s
    tl = t.replace('global', 'gobal')
    for s in sheetnames:
        if _norm(s).replace('global', 'gobal') == tl: return s
    return None

def _map_site(val):
    if val in OU_TO_SITE: return OU_TO_SITE[val]
    norm = lambda s: s.lower().replace(' ', '').replace('-', '')
    for k, v in OU_TO_SITE.items():
        if norm(k).replace('global', 'gobal') == norm(val).replace('global', 'gobal'): return v
    return val

def _to_time_str(val):
    if val is None: return None
    if isinstance(val, _dt.time): return val.strftime('%H:%M')
    if isinstance(val, _dt.datetime): return val.strftime('%H:%M')
    if isinstance(val, str):
        for fmt in ('%H:%M:%S', '%H:%M'):
            try: return _dt.datetime.strptime(val.strip(), fmt).strftime('%H:%M')
            except ValueError: continue
    return None

def _read_ou_sheet(ws, col_start, col_end):
    lbl_idx  = column_index_from_string('A')
    d_start  = column_index_from_string(col_start)
    d_end    = column_index_from_string(col_end)
    tmp_cols = [get_column_letter(c) for c in range(d_start, d_end + 1)]
    pst_list, data = [], {c: [] for c in tmp_cols}
    for row in range(ROW_START, ROW_END + 1):
        pst_list.append(_to_time_str(ws.cell(row=row, column=lbl_idx).value))
        for i, c in enumerate(range(d_start, d_end + 1)):
            v = ws.cell(row=row, column=c).value
            try: data[tmp_cols[i]].append(float(v) if v is not None else None)
            except (TypeError, ValueError): data[tmp_cols[i]].append(None)
    df = pl.DataFrame({'PST': pst_list, **data})
    df = df.filter(pl.col('PST').is_not_null() & (pl.col('PST').str.strip_chars() != 'PST'))
    return df.rename({old: new for old, new in zip(tmp_cols, DAY_METRIC_COLS[:len(tmp_cols)])})

def _load_all_ou_cached(folder, lg_keyword, nl_keyword):
    all_jobs = (
        [(p, LG_SHEET_CONFIG, 'Lodging chat')     for p in _find_all(folder, lg_keyword)] +
        [(p, NL_SHEET_CONFIG, 'Non-Lodging chat') for p in _find_all(folder, nl_keyword)]
    )
    if os.path.exists(ou_cache_path):
        df_cache = pl.read_parquet(ou_cache_path)
        cached   = set(df_cache['_src'].to_list())
        print(f'  Cache hit: {len(cached)} files already cached')
    else:
        df_cache, cached = None, set()
    new_frames = []
    for path, sheet_config, lob_label in all_jobs:
        fname = os.path.basename(path)
        if fname in cached:
            print(f'  [SKIP]  {fname}'); continue
        wb      = openpyxl.load_workbook(path, data_only=True)
        week_dt = _extract_week(path)
        print(f'  [LOAD]  {fname} | Week: {week_dt}')
        for target, (col_s, col_e) in sheet_config.items():
            actual = _resolve_sheet(wb.sheetnames, target)
            if actual is None: continue
            df = _read_ou_sheet(wb[actual], col_s, col_e).with_columns([
                pl.lit(week_dt).alias('Week'),
                pl.lit(lob_label).alias('LOB'),
                pl.lit(_map_site(actual)).alias('Site'),
                pl.lit(fname).alias('_src'),
            ])
            new_frames.append(df)
        wb.close()
    if new_frames:
        all_data = pl.concat([df_cache, *new_frames] if df_cache is not None else new_frames, how='vertical')
        os.makedirs(os.path.dirname(ou_cache_path), exist_ok=True)
        all_data.write_parquet(ou_cache_path)
        print(f'  Cache updated: {all_data["_src"].n_unique()} files total')
    else:
        all_data = df_cache
        print(f'  All files served from cache')
    return all_data.drop('_src')

print('\nLoading ALL OU Mail files (with cache)...')
all_combined = _load_all_ou_cached(input_ou_mail, 'Global OU LG Chat', 'Global OU NL Chat')

id_cols  = ['Site', 'LOB', 'Week', 'PST']
val_cols = [c for c in all_combined.columns if c not in id_cols]

df_long = (
    all_combined
    .unpivot(index=id_cols, on=val_cols, variable_name='_col', value_name='Value')
    .with_columns([
        pl.col('_col').str.split_exact('_', 1).struct.field('field_0').alias('Day'),
        pl.col('_col').str.split_exact('_', 1).struct.field('field_1').alias('OU Status'),
    ])
    .with_columns(pl.col('Day').replace_strict(DAY_OFFSET, return_dtype=pl.Int32).alias('_offset'))
    .with_columns(
        (pl.col('Week').cast(pl.Datetime('us')) + pl.duration(days=pl.col('_offset')))
        .dt.date().alias('PST Date')
    )
    .drop(['_col', 'Day', '_offset'])
)

ou_req_df = (
    df_long
    .filter(
        (pl.col('OU Status') == 'Req W') &
        pl.col('LOB').is_in(['Lodging chat', 'Non-Lodging chat']) &
        (pl.col('Site') != 'Concentrix (Global)')
    )
    .select(['LOB', 'Site', 'PST', 'PST Date', 'Value'])
    .with_columns([
        (pl.col('PST') + ':00').alias('Interval'),
        pl.concat_str([
            pl.col('PST Date').dt.month().cast(pl.String), pl.lit('/'),
            pl.col('PST Date').dt.day().cast(pl.String),   pl.lit('/'),
            pl.col('PST Date').dt.year().cast(pl.String),
        ]).alias('_date_str'),
        (pl.col('Value').cast(pl.Float64, strict=False) / 2).alias('Value'),
    ])
    .pivot(on='_date_str', index=['LOB', 'Site', 'Interval'], values='Value', aggregate_function='first')
    .rename({'LOB': 'PSP'})
)

ou_date_cols = sorted(
    [c for c in ou_req_df.columns if c not in ('PSP', 'Site', 'Interval')],
    key=lambda x: _dt.datetime.strptime(x, '%m/%d/%Y')
)
ou_req_df = (
    ou_req_df
    .with_columns([pl.col(c).cast(pl.Float64, strict=False) for c in ou_date_cols])
    .select(['PSP', 'Site', 'Interval'] + ou_date_cols)
    .sort(['PSP', 'Site', 'Interval'], maintain_order=True)
)

print(f'\n[OU_Req_File.csv] Shape: {ou_req_df.shape} | Sites: {ou_req_df["Site"].unique().to_list()}')

_ID_COLS = ['PSP', 'Site', 'Interval']

def _parse_date_header(s: str) -> _dt.date | None:
    for fmt in ('%m/%d/%Y', '%Y-%m-%d'):
        try: return _dt.datetime.strptime(s, fmt).date()
        except ValueError: continue
    return None

def _in_range(d: _dt.date) -> bool:
    if d is None: return False
    if date_from is not None and d < date_from: return False
    if date_to   is not None and d > date_to:   return False
    return True

def _filter_date_cols(df: pl.DataFrame) -> pl.DataFrame:
    keep = [c for c in df.columns if c in _ID_COLS or (
        _parse_date_header(c) is not None and _in_range(_parse_date_header(c))
    )]
    return df.select(keep)

def _unpivot_for_compare(df: pl.DataFrame, source: str) -> pl.DataFrame:
    date_cols = [
        c for c in df.columns
        if c not in _ID_COLS
        and _parse_date_header(c) is not None
        and _in_range(_parse_date_header(c))
    ]
    return (
        df.select(_ID_COLS + date_cols)
          .unpivot(index=_ID_COLS, on=date_cols, variable_name='_raw', value_name=source)
          .with_columns([
              pl.col(source).cast(pl.Float64, strict=False),
              pl.coalesce([
                  pl.col('_raw').str.to_date('%m/%d/%Y', strict=False),
                  pl.col('_raw').str.to_date('%Y-%m-%d', strict=False),
              ]).dt.strftime('%Y-%m-%d').alias('Date'),
          ])
          .drop('_raw')
    )

sheet_excal = _filter_date_cols(result_df)
sheet_ou    = _filter_date_cols(ou_req_df)

df_exc_long = _unpivot_for_compare(result_df,  'Excalibur')
df_ou_long  = _unpivot_for_compare(ou_req_df,  'OU Mail')

df_compare = (
    df_exc_long
    .join(df_ou_long, on=_ID_COLS + ['Date'], how='full', coalesce=True)
    .filter((pl.col('Site') != 'Concentrix (Global)') & pl.col('Date').is_not_null())
    .with_columns([pl.col('Excalibur').fill_null(0.0), pl.col('OU Mail').fill_null(0.0)])
    .with_columns((pl.col('Excalibur') - pl.col('OU Mail')).round(4).alias('Diff'))
    .filter(pl.col('Diff').abs() > 0.001)
    .sort(['PSP', 'Site', 'Date', 'Interval'])
)

site_order = (
    result_df.select(['PSP', 'Site'])
    .unique(maintain_order=True)
    .with_row_index('_rank')
)

sheet_ou = (
    ou_req_df
    .join(site_order, on=['PSP', 'Site'], how='left')
    .with_columns(pl.col('_rank').fill_null(9999))
    .sort(['_rank', 'Interval'], maintain_order=True)
    .drop('_rank')
)

sheet_excal = _filter_date_cols(result_df)
sheet_ou    = _filter_date_cols(sheet_ou)

import xlsxwriter
os.makedirs(os.path.dirname(output_cmp), exist_ok=True)
workbook = xlsxwriter.Workbook(output_cmp)
sheet_excal.write_excel(workbook=workbook, worksheet='req_from_excalibur')
sheet_ou   .write_excel(workbook=workbook, worksheet='req_from_ou')
df_compare .write_excel(workbook=workbook, worksheet='diff_detail')
workbook.close()

range_str = f"{date_from or 'start'} → {date_to or 'end'}"
print(f'\n[Comparison] Date range    : {range_str}')
print(f'             Rows with diff: {len(df_compare):,}')
print(f'             Sites affected: {df_compare["Site"].n_unique()}')
print(f'             Dates affected: {df_compare["Date"].n_unique()}')
print(f'             Exported to   : {output_cmp}')
display(df_compare.head(20))

[Req.csv] Shape: (336, 40) | PSP: ['Lodging chat', 'Non-Lodging chat']

Loading ALL OU Mail files (with cache)...
  Cache hit: 15 files already cached
  [SKIP]  Global OU LG Chat 07_06_26.xlsx
  [SKIP]  Global OU LG Chat 07_13_26.xlsx
  [SKIP]  Global OU LG Chat 07_20_26.xlsx
  [SKIP]  Global OU LG Chat 07_27_26.xlsx
  [SKIP]  Global OU LG Chat 08_03_26.xlsx
  [SKIP]  Global OU LG Chat 08_10_26.xlsx
  [SKIP]  Global OU LG Chat 08_17_26.xlsx
  [SKIP]  Global OU LG Chat 08_24_26.xlsx
  [SKIP]  Global OU LG Chat 08_31_26.xlsx
  [SKIP]  Global OU NL Chat 07_06_26.xlsx
  [SKIP]  Global OU NL Chat 07_13_26.xlsx
  [SKIP]  Global OU NL Chat 07_20_26.xlsx
  [SKIP]  Global OU NL Chat 07_27_26.xlsx
  [SKIP]  Global OU NL Chat 08_03_26.xlsx
  [SKIP]  Global OU NL Chat 08_10_26.xlsx
  All files served from cache

[OU_Req_File.csv] Shape: (384, 66) | Sites: ['Concentrix (Kolkata)', 'Concentrix (Cairo)', 'Concentrix (Ho Chi Minh City)', 'Concentrix (Pune)']

[Comparison] Date range    : 2026-08-01 → 

PSP,Site,Interval,Excalibur,Date,OU Mail,Diff
str,str,str,f64,str,f64,f64
"""Lodging chat""","""Concentrix (Ho Chi Minh City)""","""02:30:00""",5.705,"""2026-08-10""",4.705,1.0
"""Lodging chat""","""Concentrix (Ho Chi Minh City)""","""03:00:00""",6.265,"""2026-08-10""",4.765,1.5
"""Lodging chat""","""Concentrix (Ho Chi Minh City)""","""05:00:00""",5.575,"""2026-08-10""",3.575,2.0
"""Lodging chat""","""Concentrix (Ho Chi Minh City)""","""05:30:00""",6.33,"""2026-08-10""",4.33,2.0
"""Lodging chat""","""Concentrix (Ho Chi Minh City)""","""06:00:00""",4.895,"""2026-08-10""",4.395,0.5
…,…,…,…,…,…,…
"""Lodging chat""","""Concentrix (Ho Chi Minh City)""","""17:00:00""",5.28,"""2026-08-10""",4.28,1.0
"""Lodging chat""","""Concentrix (Ho Chi Minh City)""","""17:30:00""",5.87,"""2026-08-10""",4.87,1.0
"""Lodging chat""","""Concentrix (Ho Chi Minh City)""","""03:00:00""",4.88,"""2026-08-11""",3.38,1.5


In [14]:
def process_psp_hours(file_path):
    try:
        df_raw = pl.read_excel(source=file_path, has_header=False, infer_schema_length=0)
        header_vals = df_raw.row(0)
        new_columns = [
            val.strftime("%Y-%m-%d") if isinstance(val, datetime)
            else str(val).strip() if val is not None
            else "Unknown"
            for val in header_vals
        ]
        df_raw.columns = new_columns
        df = df_raw.slice(1).with_columns(
            pl.col("Interval").cast(pl.String)
              .str.replace(r"^.*1899-12-31\s+", "")
              .str.slice(0, 8).alias("Interval")
        )
        final_df = (
            df.unpivot(index=["LOB", "Site", "Interval"], variable_name="Date_Str", value_name="Value")
            .with_columns([
                (pl.col("Date_Str").str.slice(0, 10) + " " + pl.col("Interval"))
                .str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False)
                .dt.truncate("1m").alias("PST_Datetime"),
                pl.col("Value").cast(pl.Float64, strict=False).fill_null(0.0).alias("PSP"),
                pl.col("LOB").str.strip_chars(),
                pl.col("Site").str.strip_chars()
            ])
            .with_columns(pl.col("PSP").sum().over(["LOB", "PST_Datetime"]).alias("Total PSP"))
        )
        return final_df.select(["LOB", "Site", "PST_Datetime", "PSP", "Total PSP"])
    except Exception:
        return pl.DataFrame()

def process_productive_hours(folder_path):
    all_files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))
    dfs = []
    for file in all_files:
        df_temp = pl.read_csv(file, infer_schema_length=0, ignore_errors=True, encoding="utf-8")
        rename_map = {
            "Forecast Group Name"  : "LOB",
            "Business Location"    : "Site",
            "Interval"             : "Raw_Interval",
            "Productive Hours (Sum)": "Productive Hours"
        }
        existing_rename = {k: v for k, v in rename_map.items() if k in df_temp.columns}
        if existing_rename:
            df_temp = df_temp.rename(existing_rename)
        if "Productive Hours" in df_temp.columns:
            df_temp = df_temp.with_columns(
                pl.col("Productive Hours").cast(pl.String).str.strip_chars()
                  .str.replace_all(r",", "").str.replace_all(r"[^\d.-]", "")
                  .replace({"": "0", "-": "0", "N/A": "0", "null": "0"})
                  .cast(pl.Float64, strict=False).fill_null(0.0).alias("Productive Hours")
            )
        for col in ["LOB", "Site", "Raw_Interval"]:
            if col in df_temp.columns:
                df_temp = df_temp.with_columns(pl.col(col).cast(pl.String).str.strip_chars())
        dfs.append(df_temp)
    df_raw = pl.concat(dfs, how="vertical_relaxed")
    df = df_raw.select([pl.col("LOB"), pl.col("Site"), pl.col("Raw_Interval"), pl.col("Productive Hours")])
    df = df.with_columns([
        pl.when(pl.col("LOB") == "GEN_GEN_EN_GCS_GLG_CHT").then(pl.lit("Lodging chat"))
          .when(pl.col("LOB") == "GEN_GEN_EN_GCS_GNL_CHT").then(pl.lit("Non-Lodging chat"))
          .otherwise(pl.col("LOB")).str.strip_chars().alias("LOB"),
        pl.col("Site").str.strip_chars().alias("Site")
    ])
    df_agg = df.group_by(["LOB", "Site", "Raw_Interval"]).agg(pl.col("Productive Hours").sum())
    df_processed = df_agg.with_columns(
        pl.col("Raw_Interval").str.strip_chars()
          .str.to_datetime(format="%m/%d/%Y %H:%M", strict=False)
          .fill_null(pl.col("Raw_Interval").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False))
          .dt.truncate("1m").alias("PST_Datetime")
    ).filter(pl.col("PST_Datetime").is_not_null()).with_columns(
        pl.col("Productive Hours").sum().over(["LOB", "PST_Datetime"]).alias("Total Productive Hours")
    )
    return df_processed.select(["LOB", "Site", "PST_Datetime", "Productive Hours", "Total Productive Hours"])

def process_vendor_msp(folder_path: str) -> pl.DataFrame:
    files = glob.glob(os.path.join(folder_path, "*.csv"))
    if not files: return pl.DataFrame()
    target_columns   = ["Vendor Forecast Group Name", "Interval Time", "Forecast Productive Hour (Sum)", "Occupancy", "Staffing Attainment (Pct)", "Interval Compliance (Pct)"]
    schema_overrides = {"Forecast Productive Hour (Sum)": pl.Float64, "Occupancy": pl.Float64, "Staffing Attainment (Pct)": pl.Float64, "Interval Compliance (Pct)": pl.Float64, "Interval Time": pl.String}
    dfs = []
    for f in files:
        try: dfs.append(pl.read_csv(f, columns=target_columns, schema_overrides=schema_overrides, infer_schema_length=0))
        except Exception: continue
    if not dfs: return pl.DataFrame()
    df = pl.concat(dfs, how="vertical")
    return df.with_columns([
        pl.when(pl.col("Vendor Forecast Group Name") == "GEN_GEN_EN_GCS_GLG_CHT_Concentrix").then(pl.lit("Lodging chat"))
          .when(pl.col("Vendor Forecast Group Name") == "GEN_GEN_EN_GCS_GNL_CHT_Concentrix").then(pl.lit("Non-Lodging chat"))
          .otherwise(pl.col("Vendor Forecast Group Name")).alias("LOB"),
        pl.coalesce([
            pl.col("Interval Time").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False),
            pl.col("Interval Time").str.strptime(pl.Datetime, format="%m/%d/%Y %H:%M", strict=False),
            pl.col("Interval Time").str.strptime(pl.Datetime, format="%m/%d/%Y %I:%M:%S %p", strict=False),
        ]).alias("PST_Datetime"),
        pl.col("Forecast Productive Hour (Sum)").alias("MSP"),
    ]).select(["LOB", "PST_Datetime", "MSP", "Occupancy", "Staffing Attainment (Pct)", "Interval Compliance (Pct)"])

def calculate_msp_distribution(df):
    df = df.with_columns([
        pl.col("Productive Hours").cast(pl.Float64).fill_null(0.0),
        pl.col("Total Productive Hours").cast(pl.Float64).fill_null(0.0),
        pl.col("MSP").cast(pl.Float64).fill_null(0.0).round(4),
        pl.col("PSP").cast(pl.Float64).fill_null(0.0).round(4),
        pl.col("Total PSP").cast(pl.Float64).fill_null(0.0).round(4),
        (pl.col("PST_Datetime").dt.strftime("%H:%M") + "-" +
         (pl.col("PST_Datetime") + pl.duration(minutes=29)).dt.strftime("%H:%M")).alias("PST_Interval_Range"),
        pl.col("PST_Datetime").dt.time().alias("PST_Interval"),
        pl.col("PST_Datetime").dt.date().alias("Date"),
        (pl.col("PST_Datetime")
         .dt.replace_time_zone("America/Los_Angeles", ambiguous="earliest", non_existent="null")
         .dt.convert_time_zone("Asia/Ho_Chi_Minh")
         .dt.replace_time_zone(None).alias("VNT_Datetime"))
    ])
    df = df.with_columns([
        (pl.col("MSP") - pl.col("Total PSP")).round(4).alias("Variance (MSP-PSP)"),
        pl.when(pl.col("Site").str.contains("Ho Chi Minh")).then(pl.col("PSP")).otherwise(0.0)
          .sum().over(["LOB", "PST_Datetime"]).round(4).alias("PSP_HCM")
    ]).with_columns((pl.col("Total PSP") - pl.col("PSP_HCM")).round(4).alias("Total_PSP_Non_HCM"))
    is_lodging      = pl.col("LOB").str.to_lowercase().str.contains("lodging chat")
    is_hcm          = pl.col("Site").str.contains("Ho Chi Minh")
    is_single_site  = (pl.col("PSP") - pl.col("Total PSP")).abs() < 0.0001
    is_positive_var = pl.col("Variance (MSP-PSP)") > 0
    calc_prorated     = pl.col("Variance (MSP-PSP)") * (pl.col("PSP") / pl.col("Total PSP"))
    calc_loss_sharing = pl.col("Variance (MSP-PSP)") * (pl.col("PSP") / pl.col("Total_PSP_Non_HCM").replace(0, 1))
    df = df.with_columns(
        pl.when(is_lodging & (is_positive_var | is_single_site)).then(calc_prorated)
          .otherwise(
              pl.when(is_lodging & is_hcm).then(0.0)
                .otherwise(pl.when(is_lodging).then(calc_loss_sharing).otherwise(calc_prorated))
          ).fill_nan(0.0).fill_null(0.0).round(4).alias("Initial MSP allocation")
    )
    df = df.with_columns(
        pl.when((pl.col("PSP") + pl.col("Initial MSP allocation")) < -0.0001)
          .then(-pl.col("PSP")).otherwise(pl.col("Initial MSP allocation")).alias("Initial Adjustment")
    ).with_columns(
        (pl.col("Initial MSP allocation") - pl.col("Initial Adjustment")).alias("Pending MSP to be assigned")
    ).with_columns(
        pl.col("Pending MSP to be assigned").sum().over(["LOB", "PST_Datetime"]).alias("Total_Pending_Global")
    ).with_columns(
        pl.when(is_lodging & pl.col("Site").str.contains("Ho Chi Minh"))
          .then(pl.col("Initial Adjustment") + pl.col("Total_Pending_Global"))
          .otherwise(pl.col("Initial Adjustment")).alias("Final adjustment")
    )
    return df.with_columns((pl.col("PSP") + pl.col("Final adjustment")).alias("MSP site wise"))

def calculate_billable_logic(df):
    return (
        df.with_columns([
            pl.min_horizontal(["MSP site wise", "Productive Hours"]).alias("Billable"),
            pl.when(pl.col("Productive Hours") >= pl.col("MSP site wise"))
              .then(0.0).otherwise(pl.col("MSP site wise") - pl.col("Productive Hours")).alias("Billable Loss"),
            pl.when((pl.col("Productive Hours") - pl.col("MSP site wise")) < 0)
              .then(0.0).otherwise(pl.col("Productive Hours") - pl.col("MSP site wise")).alias("Over Production")
        ])
        .with_columns([
            pl.col("Billable Loss").sum().over(["LOB", "PST_Datetime"]).alias("Global Billable Loss"),
            pl.col("Over Production").sum().over(["LOB", "PST_Datetime"]).alias("Global Over Production")
        ])
        .with_columns(
            pl.when(pl.col("Billable Loss") > 0).then(0.0)
              .otherwise((pl.col("Global Billable Loss") * pl.col("Over Production")) / pl.col("Global Over Production"))
              .fill_nan(0.0).fill_null(0.0).alias("Compensated by other site")
        )
        .with_columns([
            pl.min_horizontal(["Over Production", "Compensated by other site"]).alias("Actual Compensation"),
            (pl.col("Billable") + pl.min_horizontal(["Over Production", "Compensated by other site"])).alias("Final Billable")
        ])
        .with_columns((pl.col("Final Billable") - pl.col("MSP site wise")).alias("Unbillable Hours"))
    )

def run_pipeline():
    req_path  = os.path.join(first_glob, r"Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\OU.xlsx")
    prod_path = os.path.join(first_glob, r"Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\INPUT_PRODUCTIVE_REVENUE")
    wfm_path  = os.path.join(first_glob, r"Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\INPUT_WORKFORCE_VENDOR")
    df_req  = process_psp_hours(req_path)
    df_prod = process_productive_hours(prod_path)
    df_msp  = process_vendor_msp(wfm_path)
    if df_req.is_empty() or df_prod.is_empty() or df_msp.is_empty(): return None
    df_merged = (
        df_req
        .join(df_prod, on=["LOB", "Site", "PST_Datetime"], how="left", coalesce=True)
        .join(df_msp,  on=["LOB", "PST_Datetime"],         how="left", coalesce=True)
    )
    df_distributed = calculate_msp_distribution(df_merged)
    df_final       = calculate_billable_logic(df_distributed)
    final_cols = [
        "LOB", "Site", "Date", "PST_Interval", "PST_Interval_Range", "PST_Datetime", "VNT_Datetime",
        "PSP", "Total PSP", "MSP", "Variance (MSP-PSP)", "PSP_HCM", "Total_PSP_Non_HCM",
        "Initial MSP allocation", "Initial Adjustment", "Pending MSP to be assigned", "Final adjustment",
        "MSP site wise", "Productive Hours", "Total Productive Hours", "Billable", "Billable Loss",
        "Over Production", "Compensated by other site", "Actual Compensation", "Final Billable",
        "Unbillable Hours", "Occupancy", "Staffing Attainment (Pct)", "Interval Compliance (Pct)"
    ]
    existing_cols = [c for c in final_cols if c in df_final.columns]
    df_result = df_final.select(existing_cols).sort(["LOB", "Site", "PST_Datetime"])
    df_result.write_excel(f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/excalibur_raw.xlsx')
    df_result.write_parquet(parquet_path)
    return df_result

df_result = run_pipeline()

In [15]:
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)

target_cols = [
    "PST_Interval", "PSP", "Total PSP", "MSP",
    "Variance (MSP-PSP)", "PSP_HCM", "Total_PSP_Non_HCM",
    "Initial MSP allocation", "Initial Adjustment",
    "Pending MSP to be assigned", "Final adjustment", "MSP site wise"
]

df_result.filter(
    (pl.col("Date") == date(2026, 6, 10)) &
    (pl.col("PST_Interval") >= time(13, 0, 0)) &
    (pl.col("Site") == "Concentrix (Ho Chi Minh City)")
).select(target_cols).sort("PST_Interval")

PST_Interval,PSP,Total PSP,MSP,Variance (MSP-PSP),PSP_HCM,Total_PSP_Non_HCM,Initial MSP allocation,Initial Adjustment,Pending MSP to be assigned,Final adjustment,MSP site wise
time,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
13:00:00,9.155,18.73,18.7289,-0.0011,9.155,9.575,0.0,0.0,0.0,0.0,9.155
13:00:00,1.16,10.47,10.9868,0.5168,1.16,9.31,0.0573,0.0573,0.0,0.0573,1.2173
13:30:00,9.275,18.73,18.7289,-0.0011,9.275,9.455,0.0,0.0,0.0,0.0,9.275
13:30:00,0.69,9.065,9.4935,0.4285,0.69,8.375,0.0326,0.0326,0.0,0.0326,0.7226
14:00:00,9.55,16.21,16.208,-0.002,9.55,6.66,0.0,0.0,0.0,0.0,9.55
14:00:00,1.47,9.355,9.8035,0.4485,1.47,7.885,0.0705,0.0705,0.0,0.0705,1.5405
14:30:00,10.275,16.2,16.1997,-0.0003,10.275,5.925,0.0,0.0,0.0,0.0,10.275
14:30:00,4.665,8.24,8.6301,0.3901,4.665,3.575,0.2209,0.2209,0.0,0.2209,4.8859
15:00:00,10.4,17.215,17.2164,0.0014,10.4,6.815,0.0008,0.0008,0.0,0.0008,10.4008


In [16]:
def view_hcm_daily_summary(df):
    print("HCM Daily Summary — Concentrix (Ho Chi Minh City)")
    df_hcm = df.filter(pl.col("Site").str.contains("Ho Chi Minh"))
    if df_hcm.is_empty():
        print("No HCM data found.")
        return
    target_cols = ["PSP", "MSP", "MSP site wise", "Final Billable", "Billable Loss"]
    valid_cols  = [c for c in target_cols if c in df_hcm.columns]
    df_daily    = df_hcm.group_by("Date").agg([pl.col(c).sum() for c in valid_cols]).sort("Date")
    df_pivot = (
        df_daily
        .unpivot(index="Date", variable_name="Metric", value_name="Value")
        .with_columns(pl.col("Date").dt.strftime("%Y-%m-%d").alias("Date_Str"))
        .pivot(values="Value", index="Metric", on="Date_Str", aggregate_function="sum", sort_columns=True)
    )
    with pl.Config(tbl_rows=20, tbl_cols=20, float_precision=2, tbl_width_chars=300):
        print(df_pivot)

if 'df_final' in locals():   view_hcm_daily_summary(df_final)
elif 'df_result' in locals(): view_hcm_daily_summary(df_result)
else: print("No df_final/df_result found. Run the pipeline cell first.")

HCM Daily Summary — Concentrix (Ho Chi Minh City)
shape: (5, 246)
┌────────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┐
│ Metric         ┆ 2025-12-29 ┆ 2025-12-30 ┆ 2025-12-31 ┆ 2026-01-01 ┆ 2026-01-02 ┆ 2026-01-03 ┆ 2026-01-04 ┆ 2026-01-05 ┆ 2026-01-06 ┆ … ┆ 2026-08-21 ┆ 2026-08-22 ┆ 2026-08-23 ┆ 2026-08-24 ┆ 2026-08-25 ┆ 2026-08-26 ┆ 2026-08-27 ┆ 2026-08-28 ┆ 2026-08-29 ┆ 2026-08-30 │
│ ---            ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│ str            ┆ f64        ┆ f64        ┆ f64        ┆ f64        ┆ f64        ┆ f64        ┆ f64        ┆ f64        ┆ f